# Trait cross-reference: persona-lsh's trait directions against hexaco-lsh's factor directions

Two libraries from one procedure, on the same models. `persona-lsh.ipynb` gives one direction per behavioral
trait (evil, sycophantic, hallucinating, optimistic, impolite, apathetic, humorous) and `hexaco-lsh.ipynb` one
per personality factor (O, C, E, A, N, H). Each is the mean of *with minus without* over the questions that
elicited it, from open replies under an instruction pair, and each separates its own set (`persona-lsh-results`
and `hexaco-lsh-results`, section 4). This notebook asks how the two sets sit against each other, per model, at
one layer: is `evil` low honesty-humility and low agreeableness, as it is in people?

**Rebuilt 2026-11-24.** The first version (2026-11-20) took its factor directions from the models' self-report
on an inventory (`persona-hexaco`, since dropped; the notebooks are in git history before this change). That run
was a null: no cosine beat the permutation null in five models, priors at chance (26 of 48), and a one-digit
self-rating and a persona reply were too different to tell "unrelated" from "not comparable" (`../PLAN.md`
section 5). With factor directions built the way the trait directions are, that excuse is gone, and the risk
is the opposite one: a cosine away from 0 can come from what the two procedures share. So this version adds
the checks for that, and question-level tests in place of the inventory ones:

| Section | Question |
|---|---|
| 1 | Which runs are in both experiments, with the same checkpoint and layer count, and how do the two designs differ? Which projection seed does each model get? How reliable is each direction (split halves)? |
| 2 | **Cosine matrix**: each trait direction against each factor direction, with a sign-flip null and a bootstrap interval over questions |
| 3 | **The shared component**: how much of every direction is one common axis, and what is left of the matrix without it? |
| 4 | **Priors**: do the signs match the human literature (evil = low H and low A, ...), with and without the shared component? |
| 5 | **Across models**: is the trait-by-factor pattern the same in every model? |
| 6 | **Poles**: question by question, which factor pole does a trait reply land on, and which trait does a factor reply land on? |
| 7 | **Span**: how much of a trait direction lies in the space of the six factor directions, and with what weights? |
| 8 | **RP-LSH**: does the cross-reference survive hashing, with the model-derived seed? |
| 9 | **Layer sweep**: where in depth is the cross-reference strongest? |

**What must match between the two experiments**, and section 1 checks:
- the **checkpoint** (same repo; on the rig, the same fingerprint from `lsh.seed`) and so the **layer count**;
- the **layer** (the same depth rule on both sides);
- dtype, transformers version, reply budget and gate judge, which are printed where they differ;
- and, for digests only, the **projection seed**, derived from the model (`lsh.seed`).

The seeds of the two draws don't need to match: generation is greedy in both, and a draw only picks questions.

Raw states don't leave the rig. This notebook loads them read-only and writes tables of cosines to
`<hexaco-lsh experiment>/results/crossref/`. CPU torch is enough.

In [ ]:
import json
import os
import sys
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap

PERSONA_LSH = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lsh" / "rp_backend.py").is_file())
sys.path.insert(0, str(PERSONA_LSH))
from lsh import RandomProjectionBackend
from lsh import seed as lsh_seed

LSH_ROOT = Path(os.environ.get("PERSONA_LSH_OUTPUT", Path.home() / "persona-lsh-output"))
LSH_EXPERIMENT = "latest"     # a directory under LSH_ROOT, or "latest": the newest with an experiment.json.
                              # The match for hexaco-lsh's first run is "rand40-t256-s52466129" (same five models,
                              # 256 tokens, gemma-qat's gate)
HXL_ROOT = Path(os.environ.get("HEXACO_LSH_OUTPUT", Path.home() / "hexaco-lsh-output"))
HXL_EXPERIMENT = "latest"     # a directory under HXL_ROOT, or "latest"
ONLY = []                     # run names; empty = every run in both

TRAIT_ORDER = ["evil", "sycophantic", "hallucinating", "optimistic", "impolite", "apathetic", "humorous"]
# hexaco-lsh names its traits in full; the factor letters are the columns here. The high pole is "with".
FACTOR = {"openness": "O", "conscientiousness": "C", "extraversion": "E", "agreeableness": "A",
          "neuroticism": "N", "honesty-humility": "H"}
FACTORS = list(FACTOR.values())

LAYER_DEPTH = 20 / 28         # persona-lsh's default depth, as a share of the layers; same rule on both sides
PERMUTATIONS, PERM_SEED = 500, 0   # sign-flip nulls per factor, and their seed
BOOT = 1000                   # bootstrap resamples of the questions on both sides, per cell
ALPHA = 0.01                  # a cell is marked when its null p is below this (min p is 1 / (PERMUTATIONS + 1))
                              # and its bootstrap interval at 1 - ALPHA excludes 0
RP_BITS = [256, 1024]
PLUS_RAND = 0                 # offset added to each model's base seed (0 = the model's seed under the salt)
REPLICATES = 5                # projections per model for section 8: PLUS_RAND, PLUS_RAND + 1, ...
SALT = None                   # None: $PERSONA_LSH_SALT or ~/.config/persona-lsh/salt; "" = public, unsalted
SEED_SCOPE = "full"           # "full": every file in the HF snapshot; "weights": weights, config, tokenizer only
FALLBACK_SEED = 42            # when the HF cache isn't here (off the rig): persona-lsh's old fixed seed
SAVE_FIGS = True

# Priors from the personality literature, as (trait, factor): expected sign. Sycophancy is flattery and
# ingratiation, which HEXACO puts at low Sincerity (H) with high Agreeableness; evil is the dark-triad corner, low
# H and low A; impolite is low A; apathetic low E and C; optimistic high E and low N; humorous high E and O.
# hallucinating has no clear prior and is left exploratory.
PRIORS = {("evil", "H"): -1, ("evil", "A"): -1, ("sycophantic", "H"): -1, ("sycophantic", "A"): +1,
          ("impolite", "A"): -1, ("apathetic", "E"): -1, ("apathetic", "C"): -1, ("optimistic", "E"): +1,
          ("optimistic", "N"): -1, ("humorous", "E"): +1, ("humorous", "O"): +1}

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)
SEQ_DIV = LinearSegmentedColormap.from_list("div", ["#104281", "#5598e7", "#f0efec", "#ec8a89", "#a52a29"])
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})


def show(df, index=False):
    print(df.to_string(index=index, float_format=lambda v: f"{v:+.2f}"))


def spearman(a, b):
    return float(pd.Series(np.asarray(a, dtype=float)).rank().corr(pd.Series(np.asarray(b, dtype=float)).rank()))


def heat(ax, table, vmax, fmt=lambda v: f"{v:+.2f}", marks=None):
    values = table.to_numpy(dtype=float)
    ax.imshow(np.ma.masked_invalid(values), cmap=SEQ_DIV, vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if np.isfinite(values[i, j]):
                star = "*" if marks is not None and bool(marks.iat[i, j]) else ""
                ax.text(j, i, fmt(values[i, j]) + star, ha="center", va="center", fontsize=7,
                        color="white" if abs(values[i, j]) > 0.6 * vmax else "#0b0b0b")
    ax.set_xticks(range(table.shape[1]), table.columns)
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.grid(False)

## 1. Load the experiments, match runs, derive seeds

A run is used when it's finished in both experiments, both name the same HF repo, and both have the same number
of layers and hidden size. Each direction uses every eliciting question of its trait or factor (test and eval
splits: the means aren't being classified against their own questions here) and only what `kept_traits.json`
kept for that run. So a model that didn't elicit a trait or factor has no row or column for it.

The design differences between the two experiments are printed. A different reply budget or gate judge doesn't
stop the comparison, but it is a reason a cosine can be smaller than it would be between matched runs.

In [ ]:
def experiment_dir(root, wanted):
    if wanted == "latest":
        found = sorted(root.glob("*/experiment.json"), key=lambda p: json.loads(p.read_text()).get("created", ""))
        assert found, f"no experiment with an experiment.json under {root}: name one"
        return found[-1].parent
    assert (root / wanted).is_dir(), f"no {root / wanted}"
    return root / wanted


def read_json(path, default=None):
    return json.loads(path.read_text()) if path.exists() else default


LSH_DIR = experiment_dir(LSH_ROOT, LSH_EXPERIMENT)
HXL_DIR = experiment_dir(HXL_ROOT, HXL_EXPERIMENT)
OUT = HXL_DIR / "results" / "crossref"
OUT.mkdir(parents=True, exist_ok=True)
KEPT = {"traits": read_json(LSH_DIR / "kept_traits.json")["runs"], "factors": read_json(HXL_DIR / "kept_traits.json")["runs"]}
DIRECTORY = {"traits": LSH_DIR, "factors": HXL_DIR}

designs = {"persona-lsh": read_json(LSH_DIR / "experiment.json", {}), "hexaco-lsh": read_json(HXL_DIR / "experiment.json", {})}
keys = ("split", "framing", "draw by", "max new tokens", "gate", "gate by")
print("persona-lsh:", LSH_DIR, "| hexaco-lsh:", HXL_DIR)
print("design differences:", {k: {n: d.get(k) for n, d in designs.items()} for k in keys
                              if designs["persona-lsh"].get(k) != designs["hexaco-lsh"].get(k)} or "none")


def finished(side, name):
    # Judged, and every kept trait captured.
    kept = KEPT[side]
    return name in kept and all((DIRECTORY[side] / name / f"all_{t}_diffs.pt").exists() for t in kept[name]["kept"])


def runs_in(side):
    return {p.name for p in DIRECTORY[side].iterdir() if p.is_dir() and finished(side, p.name)}


lsh_runs, hxl_runs = runs_in("traits"), runs_in("factors")
RUNS, rows = [], []
for name in sorted(lsh_runs | hxl_runs):
    if ONLY and name not in ONLY:
        continue
    row = {"run": name, "persona-lsh": name in lsh_runs, "hexaco-lsh": name in hxl_runs}
    if name in lsh_runs and name in hxl_runs:
        li, fi = read_json(LSH_DIR / name / "run_info.json"), read_json(HXL_DIR / name / "run_info.json")
        row |= {"repo": fi.get("repo"), "same repo": fi.get("repo") == li.get("repo"),
                "dtype": f"{li.get('dtype')} / {fi.get('dtype')}",
                "transformers": f"{li.get('transformers')} / {fi.get('transformers')}",
                "tokens": f"{li.get('max_new_tokens')} / {fi.get('max_new_tokens')}"}
        if row["same repo"]:
            RUNS.append(name)
    rows.append(row)
show(pd.DataFrame(rows))
assert RUNS, "no run is finished in both experiments"
REPO = {n: read_json(HXL_DIR / n / "run_info.json").get("repo") for n in RUNS}
print("\nruns compared:", RUNS)

In [ ]:
# Projection seeds: the model's fingerprint (its full HF metadata), salted with the project's common salt,
# plus PLUS_RAND. Off the rig the HF cache isn't here and every model falls back to FALLBACK_SEED.
SEEDS, rows = {}, []
salt = lsh_seed.load_salt(SALT)
for name in RUNS:
    try:
        from huggingface_hub import snapshot_download
        snap = snapshot_download(REPO[name], local_files_only=True)
        info = lsh_seed.model_seed(snap, plus_rand=PLUS_RAND, scope=SEED_SCOPE, salt=salt)
    except Exception as exc:
        info = {"repo": REPO[name], "fingerprint": None, "salt id": None, "base": FALLBACK_SEED,
                "plus_rand": PLUS_RAND, "seed": lsh_seed.projection_seed(FALLBACK_SEED, PLUS_RAND),
                "note": f"fallback ({type(exc).__name__})"}
    SEEDS[name] = info
    rows.append({"run": name, "fingerprint": (info["fingerprint"] or "")[:16], "salt id": info.get("salt id"),
                 "plus_rand": info["plus_rand"], "seed": info["seed"], "note": info.get("note", "")})
show(pd.DataFrame(rows))
# The record keeps the fingerprint and salt id, not the salt; with a salt, the base seed is secret too.
(OUT / "seeds.json").write_text(json.dumps({n: {k: v for k, v in s.items() if k != "base" or not salt}
                                            for n, s in SEEDS.items()}, indent=1))

### Directions, and how reliable each is

- **Trait** (persona-lsh): the mean of the eliciting questions' `with - without` vectors at the layer, as a unit
  vector. It points toward the trait.
- **Factor** (hexaco-lsh): the same, from the factor's questions. It points toward the high pole (for N, toward
  neurotic).

**Split-half reliability** is the cosine between a direction built from its test-split questions and one from
its eval-split questions (alternate questions when the experiment has no split). It is the ceiling for
everything below: a trait can't line up with a factor better than either lines up with itself, so a
cross-cosine of 0.3 means more against reliabilities of 0.6 than against 0.95.

In [ ]:
def layer_of(n_layers):
    return max(1, round(n_layers * LAYER_DEPTH))


def unit(v):
    v = np.asarray(v, dtype=np.float64)
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)


def stack(side, name, rename=None):
    # ({trait: [eliciting questions, layers+1, hidden] float32}, {trait: split of each of those questions})
    d = DIRECTORY[side] / name
    elic = read_json(d / "eliciting_questions.json")
    picks = (read_json(d / "selection.json") or {}).get("traits")
    vectors, splits = {}, {}
    for t in KEPT[side][name]["kept"]:
        x = torch.load(d / f"all_{t}_diffs.pt", map_location="cpu").float().numpy()
        rows = [i for i in elic.get(t, []) if i < len(x) and np.isfinite(x[i]).all()]
        if rows:
            key = (rename or {}).get(t, t)
            vectors[key] = x[rows]
            splits[key] = np.array([picks[t][i]["split"] == "test" if picks else k % 2 == 0 for k, i in enumerate(rows)])
    return vectors, splits


TRAIT_X, FACTOR_X, HALF, LAYERS = {}, {}, {}, {}
for name in RUNS:
    TRAIT_X[name], th = stack("traits", name)
    FACTOR_X[name], fh = stack("factors", name, rename=FACTOR)
    HALF[name] = th | fh
    shapes = {side: (next(iter(x.values())).shape[1] - 1, next(iter(x.values())).shape[2])
              for side, x in (("persona-lsh", TRAIT_X[name]), ("hexaco-lsh", FACTOR_X[name]))}
    assert len(set(shapes.values())) == 1, f"{name}: (layers, hidden) differ: {shapes}"
    LAYERS[name] = shapes["persona-lsh"][0]


def at(name, layer):
    # ({trait: [questions, hidden]}, {factor: [questions, hidden]}) at a layer, float64.
    return ({t: x[:, layer].astype(np.float64) for t, x in TRAIT_X[name].items()},
            {f: FACTOR_X[name][f][:, layer].astype(np.float64) for f in FACTORS if f in FACTOR_X[name]})


def directions(vectors):
    return {k: unit(v.mean(0)) for k, v in vectors.items()}


AT = {n: at(n, layer_of(LAYERS[n])) for n in RUNS}
DIRS = {n: tuple(map(directions, AT[n])) for n in RUNS}
TRAITS = [t for t in TRAIT_ORDER if any(t in DIRS[n][0] for n in RUNS)]

rows = []
for n in RUNS:
    for side, vectors in zip(("trait", "factor"), AT[n]):
        for k, v in vectors.items():
            a, b = v[HALF[n][k]], v[~HALF[n][k]]
            rows.append({"run": n, "side": side, "direction": k, "questions": len(v),
                         "split-half cosine": float(unit(a.mean(0)) @ unit(b.mean(0))) if len(a) and len(b) else np.nan})
RELIABILITY = pd.DataFrame(rows)
RELIABILITY.to_csv(OUT / "reliability.csv", index=False)
for n in RUNS:
    print(f"{n}: layer {layer_of(LAYERS[n])} of {LAYERS[n]}, questions per trait "
          f"{ {t: len(v) for t, v in AT[n][0].items()} }, per factor { {f: len(v) for f, v in AT[n][1].items()} }")
print("\nsplit-half cosine of each direction:")
show(RELIABILITY.pivot(index="run", columns="direction", values="split-half cosine").reindex(index=RUNS, columns=TRAITS + FACTORS),
     index=True)
REL = RELIABILITY.set_index(["run", "direction"])["split-half cosine"]

## 2. Cosine matrix, with a null and an interval

Per model, the cosine between every trait direction and every factor direction. Two random directions in a few
thousand dimensions have cosines near 0 (SD about 1/sqrt(hidden), 0.016 at 4096), but these aren't random: both
come from the same activation space, which has a few dominant axes. Two tests, and a cell is starred when it
passes both:

- **Sign-flip null** (`PERMUTATIONS` times). Each of the factor's questions gets a random sign on its `with -
  without` vector before the mean. That keeps the questions and their vectors and removes only the consistent
  direction of *with* against *without*. `p` is the share of null |cosines| at or above the observed one, with
  add-one smoothing. It asks whether the factor's contrast puts its direction near the trait, or whether any
  signed mix of the same vectors would.
- **Bootstrap interval** (`BOOT` times). The questions on both sides are resampled with replacement and the
  cosine recomputed. The interval at 1 - `ALPHA` says how far the cosine depends on which questions were drawn.

`ceiling` is the square root of the product of the two directions' split-half cosines (section 1): about the
largest cosine two noisy directions could show if they were the same direction.

Neither test removes what every `with - without` vector shares whatever the trait. That is section 3.

In [ ]:
rng = np.random.default_rng(PERM_SEED)


def null_sign_flips(diffs, n):
    # n unit directions from the questions' with - without vectors [questions, hidden], each with a random sign.
    return unit(rng.choice([-1.0, 1.0], size=(n, len(diffs))) @ diffs)


def boot_means(diffs, n):
    # n unit directions, each the mean of a resample of the questions.
    return unit(rng.multinomial(len(diffs), np.full(len(diffs), 1 / len(diffs)), size=n).astype(np.float64) @ diffs)


def cosine_table(name, traits, factors, label):
    # One row per trait-by-factor cell: cosine, null p, bootstrap interval. `traits` and `factors` are
    # {name: [questions, hidden]}.
    rows = []
    td, fd = directions(traits), directions(factors)
    tb = {t: boot_means(v, BOOT) for t, v in traits.items()}
    for f, fv in factors.items():
        null, fb = null_sign_flips(fv, PERMUTATIONS), boot_means(fv, BOOT)
        for t in traits:
            obs, ns = float(td[t] @ fd[f]), null @ td[t]
            lo, hi = np.percentile((tb[t] * fb).sum(1), [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
            p = (int((np.abs(ns) >= abs(obs)).sum()) + 1) / (len(ns) + 1)
            rows.append({"run": name, "matrix": label, "trait": t, "factor": f, "cosine": obs, "null sd": float(ns.std()),
                         "p": p, "lo": lo, "hi": hi, "marked": bool(p < ALPHA and (lo > 0 or hi < 0))})
    return rows


def draw_matrices(table, title, path):
    fig, axes = plt.subplots(1, len(RUNS), figsize=(3.1 * len(RUNS), 0.38 * len(TRAITS) + 1.6), squeeze=False)
    vmax = max(0.2, float(table["cosine"].abs().max()))
    for ax, name in zip(axes[0], RUNS):
        mine = table[table["run"] == name]
        values = mine.pivot(index="trait", columns="factor", values="cosine").reindex(index=TRAITS, columns=FACTORS)
        marks = mine.pivot(index="trait", columns="factor", values="marked").reindex(index=TRAITS, columns=FACTORS).fillna(False)
        heat(ax, values, vmax, marks=marks)
        ax.set_title(name, fontsize=9)
        if ax is not axes[0][0]:
            ax.set_yticks([])
    fig.suptitle(title, x=0.01, ha="left", fontsize=11, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    if SAVE_FIGS:
        fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()


def pooled_tables(table):
    pooled = table.groupby(["trait", "factor"]).agg(
        cosine=("cosine", "mean"), marked=("marked", "sum"), runs=("marked", "size"),
        same_sign=("cosine", lambda c: int((np.sign(c) == np.sign(c.mean())).sum()))).reset_index()
    print("mean cosine over runs:\n")
    show(pooled.pivot(index="trait", columns="factor", values="cosine").reindex(index=TRAITS, columns=FACTORS), index=True)
    print("\nruns marked / runs with the cell:\n")
    cell = pooled.assign(cell=pooled["marked"].astype(str) + "/" + pooled["runs"].astype(str))
    print(cell.pivot(index="trait", columns="factor", values="cell").reindex(index=TRAITS, columns=FACTORS).to_string())
    steady = pooled[(pooled["runs"] == len(RUNS)) & (pooled["same_sign"] == len(RUNS))].sort_values("cosine", key=abs, ascending=False)
    print(f"\ncells with the same sign in all {len(RUNS)} runs ({len(steady)} of {len(pooled)}):",
          ", ".join(f"{r.trait}-{r.factor} {r.cosine:+.2f}" for r in steady.itertuples()) or "none")


COS = pd.DataFrame([row for name in RUNS for row in cosine_table(name, *AT[name], "raw")])
COS["ceiling"] = [np.sqrt(max(REL.get((r.run, r.trait), np.nan), 0) * max(REL.get((r.run, r.factor), np.nan), 0))
                  for r in COS.itertuples()]
COS.to_csv(OUT / "cosines.csv", index=False)
draw_matrices(COS, f"Trait direction vs factor direction, cosine (layer at {LAYER_DEPTH:.0%} depth; "
                   f"* = null p < {ALPHA} and interval excludes 0)", OUT / "2_cosines.png")
pooled_tables(COS)
print(f"\nmarked cells: {int(COS['marked'].sum())} of {len(COS)}; mean |cosine| {COS['cosine'].abs().mean():.2f}, "
      f"mean ceiling {COS['ceiling'].mean():.2f}, mean null SD {COS['null sd'].mean():.3f}")

## 3. The shared component

Every direction here is *with minus without* under an instruction pair, so the two libraries can agree for a
reason that is no trait: a component common to all of them. Candidates: following a persona instruction at all,
replying in a marked register against a plain one, or departing from the assistant's default. In hexaco-lsh the
*with* pole is near the default for five factors and the departure for neuroticism; in persona-lsh *with* is the
departure for most traits. So a default-against-departure axis would make most trait-by-factor cosines
**negative**, and positive for N, whatever the traits are.

- **The common direction** is the first principal axis of the model's unit directions, traits and factors
  together (not centered), oriented to sum positive. `loading` is each direction's cosine with it, and `share`
  the part of all the directions' variance on it (1 / their number if they were orthogonal).
- **Residual matrix**: section 2 again after projecting the common direction out of every question's vector.
  A cell that keeps its sign and its star without the common direction is a relation between that trait and that
  factor. A cell that disappears was the shared component.

Projecting out the first axis is a blunt control: it also removes whatever real relation is spread over many
cells (if most traits here are in truth low-agreeableness, that is on the first axis too). So read the two
matrices together: the raw one is the upper bound on the cross-reference, the residual one the lower.

In [ ]:
def common_direction(td, fd):
    M = np.stack(list(td.values()) + list(fd.values()))
    values, vectors = np.linalg.eigh(M @ M.T)
    c = unit(vectors[:, -1] @ M)
    c = c if (M @ c).sum() >= 0 else -c
    return c, values[-1] / len(M)


def without(vectors, c):
    return {k: v - np.outer(v @ c, c) for k, v in vectors.items()}


rows, load_rows, COMMON = [], [], {}
for name in RUNS:
    td, fd = DIRS[name]
    c, share = common_direction(td, fd)
    COMMON[name] = c
    load_rows.append({"run": name, "share": share, "if orthogonal": 1 / (len(td) + len(fd)),
                      **{k: float(v @ c) for k, v in (td | fd).items()}})
    traits, factors = AT[name]
    rows += cosine_table(name, without(traits, c), without(factors, c), "residual")
LOAD = pd.DataFrame(load_rows)
RES = pd.DataFrame(rows)
LOAD.to_csv(OUT / "common_direction.csv", index=False)
RES.to_csv(OUT / "cosines_residual.csv", index=False)

print("loading of each direction on the model's common direction, and its share of the directions' variance:\n")
show(LOAD[["run", "share", "if orthogonal"] + [c for c in TRAITS + FACTORS if c in LOAD]])
draw_matrices(RES, "The same without each model's common direction (residual cosines)", OUT / "3_cosines_residual.png")
pooled_tables(RES)

both = COS.merge(RES, on=["run", "trait", "factor"], suffixes=("", " residual"))
kept_sign = np.sign(both["cosine"]) == np.sign(both["cosine residual"])
print(f"\nraw against residual: {int(kept_sign.sum())} of {len(both)} cells keep their sign; "
      f"marked {int(both['marked'].sum())} raw, {int(both['marked residual'].sum())} residual, "
      f"{int((both['marked'] & both['marked residual'] & kept_sign).sum())} in both with one sign; "
      f"rho between the two matrices {spearman(both['cosine'], both['cosine residual']):+.2f}")
ROBUST = both[both["marked"] & both["marked residual"] & kept_sign]

## 4. Do the signs match the priors?

The `PRIORS` table (top of the notebook) lists the cells where the human literature predicts a sign. Per model:
how many of those cells have the predicted sign (chance 50% each), and how many are also marked, in the raw and
in the residual matrix. A one-sided sign test over all models' prior cells gives the overall p. Matching signs
inside the null is weak; matching signs that are marked in both matrices is the claim.

In [ ]:
def prior_table(table):
    rows = []
    for (t, f), sign in PRIORS.items():
        for r in table[(table["trait"] == t) & (table["factor"] == f)].itertuples():
            ok = bool(np.sign(r.cosine) == sign)
            rows.append({"run": r.run, "trait": t, "factor": f, "expected": "+" if sign > 0 else "-",
                         "cosine": r.cosine, "p": r.p, "sign ok": ok, "sign ok, marked": ok and r.marked})
    return pd.DataFrame(rows)


PRIOR = {}
for label, table in (("raw", COS), ("residual", RES)):
    P = PRIOR[label] = prior_table(table)
    P.to_csv(OUT / f"priors_{label}.csv", index=False)
    if not len(P):
        continue
    n, k = len(P), int(P["sign ok"].sum())
    p = sum(comb(n, i) for i in range(k, n + 1)) / 2**n
    print(f"{label}: {k} of {n} prior cells have the expected sign (one-sided sign test p = {p:.2g}); "
          f"{int(P['sign ok, marked'].sum())} of them marked")
    show(P.groupby("run", sort=False).agg(cells=("sign ok", "size"), sign_ok=("sign ok", "sum"),
                                         sign_ok_marked=("sign ok, marked", "sum")).reset_index())
    print()
print("(cells within a model aren't independent: read p as a rough guide)\n")
if len(PRIOR["raw"]):
    per = PRIOR["raw"].groupby(["trait", "factor", "expected"], sort=False).agg(
        cosine=("cosine", "mean"), runs_ok=("sign ok", "sum"), runs=("sign ok", "size")).reset_index()
    res = PRIOR["residual"].groupby(["trait", "factor"], sort=False).agg(
        residual=("cosine", "mean"), residual_ok=("sign ok", "sum")).reset_index()
    show(per.merge(res, on=["trait", "factor"]))

## 5. Is the pattern the same in every model?

Per pair of models, the Spearman correlation of their trait-by-factor cosines over the cells both have, for the
raw and the residual matrix. High values mean a shared geometry: one cross-reference table could then serve every
model. A raw rho far above the residual one says the agreement between models is mostly the shared component.

In [ ]:
rows = []
for i, a in enumerate(RUNS):
    for b in RUNS[i + 1:]:
        row = {"a": a, "b": b}
        for label, table in (("raw", COS), ("residual", RES)):
            A = table[table["run"] == a].set_index(["trait", "factor"])["cosine"]
            B = table[table["run"] == b].set_index(["trait", "factor"])["cosine"]
            cells = A.index.intersection(B.index)
            row |= {"cells": len(cells), f"rho, {label}": spearman(A[cells], B[cells]) if len(cells) >= 3 else np.nan}
        rows.append(row)
PAIRS = pd.DataFrame(rows)
PAIRS.to_csv(OUT / "model_pairs.csv", index=False)
if len(PAIRS):
    show(PAIRS)
    print(f"\nmedian rho over pairs: raw {PAIRS['rho, raw'].median():+.2f}, residual {PAIRS['rho, residual'].median():+.2f}")

## 6. Poles: where does each reply land in the other library?

Sections 2-4 compare means. This is the test the two libraries were built for, question by question: take one
trait question's `with - without` vector and find the nearest of the **twelve factor poles** (each factor
direction and its negative), by cosine. If `evil` is low honesty-humility, most `evil` replies land on `H-`.
And the reverse: each factor question against the **fourteen trait poles**.

- Chance is 1/12 (or 1 over the poles that run has). `top pole` is where most of the trait's questions go,
  `share` how many, pooled over runs, and `runs` in how many runs it is the top pole.
- `margin` is the mean gap between the best and the second-best pole's cosine: near 0, the assignment is a coin
  toss between two poles.
- No question is in the means it is compared with (the two libraries have different questions), so there is
  nothing to leave out.

A trait whose replies land on one pole in every model is a trait the factor library can name. A trait spread
over many poles isn't on any of the six axes.

In [ ]:
def nearest_poles(vectors, dirs):
    # Each question's nearest pole among +-dirs, by cosine: (pole labels, margin to the second-best pole).
    names = list(dirs)
    M = np.stack([dirs[k] for k in names])
    S = unit(vectors) @ M.T
    both = np.concatenate([S, -S], axis=1)
    labels = [f"{k}+" for k in names] + [f"{k}-" for k in names]
    order = np.sort(both, axis=1)
    return [labels[i] for i in both.argmax(1)], order[:, -1] - order[:, -2]


rows = []
for name in RUNS:
    (traits, factors), (td, fd) = AT[name], DIRS[name]
    for side, sets, dirs in (("trait", traits, fd), ("factor", factors, td)):
        for k, v in sets.items():
            poles, margin = nearest_poles(v, dirs)
            rows += [{"run": name, "side": side, "from": k, "pole": p, "margin": m, "chance": 1 / (2 * len(dirs))}
                     for p, m in zip(poles, margin)]
POLES = pd.DataFrame(rows)
POLES.to_csv(OUT / "poles.csv", index=False)


def pole_summary(side, order):
    mine = POLES[POLES["side"] == side]
    out = []
    for k in order:
        g = mine[mine["from"] == k]
        if not len(g):
            continue
        top = g["pole"].value_counts(normalize=True)
        per_run = g.groupby("run")["pole"].agg(lambda s: s.value_counts().idxmax())
        second = top.index[1] if len(top) > 1 else ""
        out.append({"from": k, "questions": len(g), "top pole": top.index[0], "share": top.iloc[0],
                    "runs": f"{int((per_run == top.index[0]).sum())}/{len(per_run)}",
                    "second": f"{second} {top.iloc[1]:.0%}" if second else "", "chance": g["chance"].mean(),
                    "margin": g["margin"].mean()})
    return pd.DataFrame(out)


def pole_figure(side, order, poles, path, title):
    mine = POLES[POLES["side"] == side]
    share = pd.crosstab(mine["from"], mine["pole"], normalize="index").reindex(index=[k for k in order if k in set(mine["from"])],
                                                                               columns=poles, fill_value=0)
    fig, ax = plt.subplots(figsize=(0.55 * len(poles) + 2.2, 0.4 * len(share) + 1.4))
    ax.imshow(share.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(share.shape[0]):
        for j in range(share.shape[1]):
            v = share.iat[i, j]
            if v >= 0.05:
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7, color="white" if v > 0.55 else "#0b0b0b")
    ax.set_xticks(range(len(poles)), poles, rotation=45 if len(poles[0]) > 3 else 0, ha="right" if len(poles[0]) > 3 else "center")
    ax.set_yticks(range(len(share)), share.index)
    ax.set_title(title)
    fig.tight_layout()
    if SAVE_FIGS:
        fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()


factor_poles = [f"{f}{s}" for f in FACTORS for s in "+-"]
trait_poles = [f"{t}{s}" for t in TRAITS for s in "+-"]
pole_figure("trait", TRAITS, factor_poles, OUT / "6_trait_poles.png",
            f"Trait questions: share landing on each factor pole (all runs; chance {1 / len(factor_poles):.0%})")
print("each trait's questions among the factor poles:\n")
T_POLES = pole_summary("trait", TRAITS)
print(T_POLES.to_string(index=False, formatters={"share": "{:.0%}".format, "chance": "{:.0%}".format, "margin": "{:.3f}".format}))
pole_figure("factor", FACTORS, trait_poles, OUT / "6_factor_poles.png",
            f"Factor questions: share landing on each trait pole (all runs; chance {1 / len(trait_poles):.0%})")
print("each factor's questions among the trait poles:\n")
F_POLES = pole_summary("factor", FACTORS)
print(F_POLES.to_string(index=False, formatters={"share": "{:.0%}".format, "chance": "{:.0%}".format, "margin": "{:.3f}".format}))

## 7. Span: how much of a trait is in the space of the six factors?

A trait can be a mix: `evil` as low H *and* low A, neither cosine large alone. Per model, the least-squares
weights that best rebuild the unit trait direction from the factor directions, and `R2`, the share of the trait
direction inside their span.

- **Null**: the same with every factor direction rebuilt from sign-flipped questions (section 2's null), so the
  span has the same vectors in it but no factor contrast. `p` is the share of null R2 at or above the observed.
- **The other way**: each factor in the span of the trait directions. If the factors are mostly inside the trait
  span and not the reverse, the trait library is the richer one.
- A few thousand dimensions are a large space: six random directions hold about 6 / hidden of any vector
  (0.15% at 4096). An R2 of 0.2 is far above that, and still leaves 80% of the trait outside the factors.

In [ ]:
def span(target, basis):
    # (R2, weights) of the unit vector `target` in the span of the rows of `basis`.
    w, *_ = np.linalg.lstsq(basis.T, target, rcond=None)
    return float(np.sum((basis.T @ w) ** 2)), w


rows = []
for name in RUNS:
    (traits, factors), (td, fd) = AT[name], DIRS[name]
    for side, targets, basis_sets, basis_dirs in (("trait in factors", td, factors, fd), ("factor in traits", fd, traits, td)):
        names = list(basis_dirs)
        B = np.stack([basis_dirs[k] for k in names])
        nulls = np.stack([null_sign_flips(basis_sets[k], PERMUTATIONS) for k in names], axis=1)   # [perm, basis, hidden]
        for k, v in targets.items():
            r2, w = span(v, B)
            null_r2 = np.array([span(v, nb)[0] for nb in nulls])
            rows.append({"run": name, "side": side, "direction": k, "R2": r2, "null R2": float(null_r2.mean()),
                         "p": (int((null_r2 >= r2).sum()) + 1) / (len(null_r2) + 1), **dict(zip(names, w))})
SPAN = pd.DataFrame(rows)
SPAN.to_csv(OUT / "span.csv", index=False)
for side, order, cols in (("trait in factors", TRAITS, FACTORS), ("factor in traits", FACTORS, TRAITS)):
    mine = SPAN[SPAN["side"] == side]
    mean = mine.groupby("direction")[["R2", "null R2"] + [c for c in cols if c in mine]].mean().reindex(order).dropna(how="all")
    mean.insert(2, "runs p<alpha", mine.groupby("direction")["p"].agg(lambda p: f"{int((p < ALPHA).sum())}/{len(p)}").reindex(mean.index))
    print(f"{side}: R2 and weights, mean over runs\n")
    show(mean, index=True)
    print()
print("R2 per run, trait in factors:\n")
show(SPAN[SPAN["side"] == "trait in factors"].pivot(index="run", columns="direction", values="R2").reindex(index=RUNS, columns=TRAITS),
     index=True)

## 8. RP-LSH: does the cross-reference survive hashing?

Each direction is hashed with `RandomProjectionBackend` at each size in `RP_BITS`, with the model's own seed
from section 1 and `REPLICATES` consecutive plusRand offsets (independent projections). The cosine is estimated
from the normalized Hamming distance h as cos(pi h). Reported per size: the mean absolute error against the exact
cosine over all trait-by-factor cells, the share of cells with the same sign, and the share of the marked cells
whose sign survives. The spread over replicates says how much the result depends on the hyperplanes drawn.

The expected error is set by the bits, not the model: the SD of the estimate near cosine 0 is about
pi x sqrt(1/(4n)), 0.098 at 256 bits and 0.049 at 1024 (mean absolute error about 0.8 of that). Cosines below
that are below the digest's resolution. The last table hashes the questions too: section 6's nearest factor pole
for each trait question, by Hamming distance, against the pole exact cosine picks.

In [ ]:
rows, pole_rows = [], []
for name in RUNS:
    (traits, factors), (td, fd) = AT[name], DIRS[name]
    dim = next(iter(td.values())).shape[0]
    exact = COS[COS["run"] == name].set_index(["trait", "factor"])
    names = list(fd)
    exact_pole = {t: nearest_poles(v, fd)[0] for t, v in traits.items()}
    for bits in RP_BITS:
        for k, s in enumerate(lsh_seed.replicate_seeds(SEEDS[name]["seed"], REPLICATES)):
            h = RandomProjectionBackend(dim=dim, n_bits=bits, seed=s)
            dt = {t: h.hash_vector(v) for t, v in td.items()}
            df = {f: h.hash_vector(v) for f, v in fd.items()}
            est = pd.Series({(t, f): np.cos(np.pi * h.distance(dt[t], df[f])) for t in dt for f in df})
            ex, sig = exact.loc[est.index, "cosine"], exact.loc[est.index, "marked"]
            rows.append({"run": name, "bits": bits, "replicate": k, "seed": s,
                         "mean abs error": float((est - ex).abs().mean()),
                         "same sign": float((np.sign(est) == np.sign(ex)).mean()),
                         "marked cells kept": float((np.sign(est[sig]) == np.sign(ex[sig])).mean()) if sig.any() else np.nan})
            if k == 0:   # the questions' digests against the factor digests, first replicate only
                same = n = 0
                for t, v in traits.items():
                    for q, pole in zip(v, exact_pole[t]):
                        d = np.array([h.distance(h.hash_vector(q), df[f]) for f in names])
                        best = int(np.abs(d - 0.5).argmax())    # farthest from 0.5: the nearest pole of either sign
                        same += f"{names[best]}{'+' if d[best] < 0.5 else '-'}" == pole
                        n += 1
                pole_rows.append({"run": name, "bits": bits, "questions": n, "same pole as cosine": same / n})
RP = pd.DataFrame(rows)
RP.to_csv(OUT / "rp_lsh.csv", index=False)
show(RP.groupby(["run", "bits"]).agg(err=("mean abs error", "mean"), err_sd=("mean abs error", "std"),
                                     same_sign=("same sign", "mean"), marked_kept=("marked cells kept", "mean")).reset_index())
for bits in RP_BITS:
    print(f"expected SD near 0 at {bits} bits: {np.pi * np.sqrt(1 / (4 * bits)):.3f}")
print("\ntrait questions: nearest factor pole by digest against by cosine\n")
show(pd.DataFrame(pole_rows))

## 9. Layer sweep

Sections 2 and 4 at every layer (cosine only, no null, to keep it quick): the mean |cosine| over all
trait-by-factor cells, and the share of prior cells with the expected sign. Depth is layer / layers so models of
different depth line up; the dashed line is the default depth.

In [ ]:
curves, rows = {}, []
for name in RUNS:
    curve = []
    for layer in range(1, LAYERS[name] + 1):
        td, fd = map(directions, at(name, layer))
        cells = [(t, f, float(td[t] @ fd[f])) for t in td for f in fd]
        prior = [np.sign(c) == PRIORS[(t, f)] for t, f, c in cells if (t, f) in PRIORS]
        curve.append({"layer": layer, "depth": layer / LAYERS[name], "mean |cos|": float(np.mean([abs(c) for *_, c in cells])),
                      "prior signs ok": float(np.mean(prior)) if prior else np.nan})
    curves[name] = curve
    best = max(curve, key=lambda c: (np.nan_to_num(c["prior signs ok"]), c["mean |cos|"]))
    rows.append({"run": name, "best layer (priors)": best["layer"], "depth": best["depth"],
                 "prior signs ok there": best["prior signs ok"],
                 "at default": curve[layer_of(LAYERS[name]) - 1]["prior signs ok"]})
show(pd.DataFrame(rows))
(OUT / "layer_curves.json").write_text(json.dumps(curves, indent=1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for name in RUNS:
    for ax, key in zip(axes, ("mean |cos|", "prior signs ok")):
        ax.plot([c["depth"] for c in curves[name]], [c[key] for c in curves[name]], marker="o", markersize=3, label=name)
for ax, key in zip(axes, ("mean |cos|", "prior signs ok")):
    ax.axvline(LAYER_DEPTH, color="#898781", linestyle=(0, (3, 2)), linewidth=1)
    ax.set_title(key)
    ax.set_xlabel("depth (layer / layers)")
axes[1].axhline(0.5, color="#c3c2b7", linewidth=1)
axes[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
if SAVE_FIGS:
    fig.savefig(OUT / "9_layer_sweep.png", bbox_inches="tight", dpi=150)
plt.show()

## What to read off

Not yet run on hexaco-lsh output: these are the questions, in the order to read them.

1. **Section 1**: the split-half cosines. They set the ceiling, and a direction below about 0.5 (few questions,
   or a factor a model barely elicits, such as Nemotron-Nano's conscientiousness) shouldn't be read at all.
2. **Section 3 before section 2**: the common direction's share and loadings. If the factors load one way and
   neuroticism the other, with the traits opposite the factors, the raw matrix is mostly default against
   departure, and the residual matrix is the one to quote.
3. **Sections 2 and 3**: the cells marked in both matrices with one sign (`ROBUST`), and of those, the ones with
   the same sign in every model. That list is the cross-reference.
4. **Section 4**: the prior cells. evil-H and evil-A are the headline pair; sycophantic (H down, A up) is the
   one prior that a single desirability axis can't produce, since it asks for opposite signs on two desirable
   poles.
5. **Section 5**: whether one table serves every model.
6. **Section 6**: which traits the factor library can name from a single reply, and whether any factor pole
   collects everything (then it is the shared component again, seen question by question).
7. **Section 7**: whether a trait is a mix of factors, and which library spans the other.
8. **Section 8**: whether 256 bits resolve the marked cells and the poles, or whether this needs 1024.
9. **Section 9**: whether the cross-reference peaks where each library's own signal did.

**The first version's run** (2026-11-20, self-report factors, five models) is written up in `../PLAN.md` section
5: no cell beyond the null, priors at chance (26 of 48), no pattern shared across models.

Limits: one draw on each side; six factors and seven traits from hand-written instruction pairs, so a cosine is
also a statement about how those pairs were worded; and the residual matrix removes one axis chosen from these
thirteen directions themselves, not an independently measured one. A direct control would be a third library of
instruction pairs with no trait in them.